In [ ]:
# Scenario: AI Research Assistant for a Corporate Innovation Team
# Imagine you’re part of a corporate innovation lab that constantly reviews new AI research papers to stay ahead of
# trends. The team struggles with long PDFs full of technical jargon, and they want a quick way to ask natural questions
# about the papers instead of reading them cover to cover.
# How the RAG Chatbot Fits In
# - Input Source: The team uploads a research paper (e.g., ai_research.pdf).
# - Chunking: The chatbot splits the paper into manageable sections so no detail is lost.
# - Embeddings + Vector DB: Each section is converted into embeddings and stored in Chroma, making the paper searchable by meaning rather than keywords.
# - Retriever: When someone asks, “What does this paper say about reinforcement learning?”, the retriever pulls the most relevant chunks.
# - LLM Response: The Hugging Face model (Flan-T5) generates a concise, human-readable answer using those chunks as context.
# - Chat Loop: The team can keep asking questions interactively, like a research assistant that knows the paper inside out.
# ==========================================================
# SIMPLE RAG CHATBOT (STABLE VERSION)
# ==========================================================

# STEP 1 — Install Libraries
!pip -q install chromadb sentence-transformers pypdf transformers


# STEP 2 — Import Libraries
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline
from google.colab import files


# STEP 3 — Upload PDF
uploaded = files.upload()
pdf_file = list(uploaded.keys())[0]

print("Loading PDF...")

reader = PdfReader(pdf_file)

text = ""
for page in reader.pages:
    content = page.extract_text()
    if content:
        text += content

print("Document Loaded")
print("Total Characters:", len(text))


# STEP 4 — Chunk the Text
def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total Chunks:", len(chunks))


# STEP 5 — Load Embedding Model
print("Loading Embedding Model...")

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding Model Ready")


# STEP 6 — Create Vector Database
client = chromadb.Client()

try:
    client.delete_collection("rag_collection")
except:
    pass

collection = client.create_collection("rag_collection")

print("Vector DB Ready")


# STEP 7 — Store Chunks in Vector DB
print("Storing embeddings...")

for i, chunk in enumerate(chunks):

    embedding = embedding_model.encode(chunk).tolist()

    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("Chunks stored successfully")


# STEP 8 — Load Language Model
print("Loading LLM...")

generator = pipeline(
    "text-generation",
    model="gpt2",
    max_new_tokens=120,
    temperature=0.3
)

print("LLM Ready")


# STEP 9 — Retrieval Function
def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


# STEP 10 — Answer Function
def answer_question(question):

    docs = retrieve(question)

    context = " ".join(docs)

    prompt = f"""
Use the following context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

    result = generator(prompt)

    return result[0]["generated_text"].replace(prompt, "")


# STEP 11 — Chat Loop
print("\n==============================")
print("RAG Chatbot Ready")
print("Type 'exit' to stop")
print("==============================\n")

while True:

    q = input("Ask a question: ")

    if q.lower() == "exit":
        print("Session ended.")
        break

    ans = answer_question(q)

    print("\nAnswer:\n", ans)
    print("\n" + "-"*50 + "\n")

Saving ai_research.pdf to ai_research (9).pdf
Loading PDF...
Document Loaded
Total Characters: 2812
Total Chunks: 7
Loading Embedding Model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding Model Ready
Vector DB Ready
Storing embeddings...
Chunks stored successfully
Loading LLM...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...


Answer:
 
Machine learning models are generally based on a set of rules and algorithms that are

based on a set of rules and algorithms that are based on a set of rules and algorithms that are based on a set of rules and algorithms that are based on

rules and algorithms that are based on rules and algorithms that are based on rules and algorithms that are based on rules and algorithms that are based on rules and algorithms that are based on rules and algorithms that are

based on rules and algorithms that are based on rules and algorithms that are based on rules and algorithms that are based on rules and algorithms

--------------------------------------------------

Ask a question: Summarize the research paper.


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
Machine Learning is a sub-question.

The sub-question is:

How do we get machine learning to be more efficient?

Machine Learning is a sub-question.

The sub-question is:

How do we get machine learning to be more efficient?

Machine Learning is a sub-question.

The sub-question is:

How do we get machine learning to be more efficient?

Machine Learning is a sub-question.

The sub-question is:

How do we get machine learning to be

--------------------------------------------------

Ask a question: Summarize the document.


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
The document is a question. A question is a sequence of words that can be answered by

using a given word. The question is a sequence of words that can be answered by

using a given word.

The question is a sequence of words that can be answered by using a given word.

The question is a sequence of words that can be answered by using a given word.

The question is a sequence of words that can be answered by using a given word.

The question is a sequence of words that can be answered by using a given word

--------------------------------------------------

Ask a question: exit
Session ended.


In [ ]:
# Scenario: Legal Research Assistant for a Corporate Compliance Team
# Context
# A corporate compliance department constantly reviews lengthy legal documents, regulatory filings, and policy updates. These documents are dense, full of
# legal terminology, and often hundreds of pages long. The team struggles to quickly extract relevant clauses or understand implications without spending hours reading.
# How the RAG Chatbot Fits In
# - Input Source: The team uploads a legal document (e.g., data_privacy_regulation.pdf).
# - Chunking: The chatbot splits the document into sections (clauses, articles, sub-sections) so no detail is overlooked.
# - Embeddings + Vector DB: Each section is converted into embeddings and stored in Chroma, enabling semantic search rather than keyword-only lookup.
# - Retriever: When someone asks, “What does this regulation say about cross-border data transfers?”, the retriever surfaces the most relevant clauses.
# - LLM Response: A Hugging Face model (e.g., Flan-T5) generates a concise, plain-language summary of those clauses, stripping away heavy legal jargon.
# - Chat Loop: The compliance team can continue asking questions interactively, like “Does this regulation conflict with GDPR?” or “What penalties are mentioned
#  for non-compliance?”.
# Outcome
# The chatbot acts as a legal research assistant, helping the compliance team quickly interpret complex documents, identify risks, and prepare summaries for executives
#  without needing to manually parse every page.


# ==========================================================
# LEGAL RESEARCH ASSISTANT (RAG CHATBOT)
# ==========================================================

# STEP 1 — Install Libraries
!pip -q install chromadb sentence-transformers pypdf transformers


# STEP 2 — Import Libraries
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline
from google.colab import files


# STEP 3 — Upload Legal Document
uploaded = files.upload()
pdf_file = list(uploaded.keys())[0]

print("Loading Legal Document...")

reader = PdfReader(pdf_file)

text = ""
for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text

print("Document Loaded")
print("Total Characters:", len(text))


# STEP 4 — Chunk the Document
def chunk_text(text, chunk_size=600, overlap=100):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total Chunks Created:", len(chunks))


# STEP 5 — Load Embedding Model
print("Loading embedding model...")

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model ready")


# STEP 6 — Create Vector Database
client = chromadb.Client()

try:
    client.delete_collection("legal_collection")
except:
    pass

collection = client.create_collection("legal_collection")

print("Vector database ready")


# STEP 7 — Store Chunks in Vector DB
print("Storing document sections...")

for i, chunk in enumerate(chunks):

    embedding = embedding_model.encode(chunk).tolist()

    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All sections stored successfully")


# STEP 8 — Load Language Model
print("Loading LLM...")

generator = pipeline(
    "text-generation",
    model="gpt2",
    max_new_tokens=150,
    temperature=0.3
)

print("LLM ready")


# STEP 9 — Retriever Function
def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


# STEP 10 — Answer Function
def answer_question(question):

    docs = retrieve(question)

    context = " ".join(docs)

    prompt = f"""
You are a legal research assistant helping a corporate compliance team.

Use the following legal document context to answer the question.

Context:
{context}

Question:
{question}

Answer in simple language:
"""

    result = generator(prompt)

    return result[0]["generated_text"].replace(prompt, "")


# STEP 11 — Chat Loop
print("\n===================================")
print("Legal Research Assistant Ready")
print("Type 'exit' to stop")
print("===================================\n")

while True:

    q = input("Ask a legal question: ")

    if q.lower() == "exit":
        print("Session ended.")
        break

    ans = answer_question(q)

    print("\nAnswer:\n", ans)
    print("\n" + "-"*60 + "\n")

Saving data_privacy_regulation.pdf to data_privacy_regulation (1).pdf
Loading Legal Document...
Document Loaded
Total Characters: 2885
Total Chunks Created: 6
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model ready
Vector database ready
Storing document sections...
All sections stored successfully
Loading LLM...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...


Answer:
 
This regulation is intended to provide a framework for the

enforcement of data protection and compliance requirements.

Article 7: Compliance and Audits

Organizations must maintain documentation demonstrating compliance with this

regulation.

Regulatory authorities may conduct periodic audits to ensure compliance with

this regulation.

Article 8: Compliance and Audits

Organizations must maintain documentation demonstrating compliance with this

regulation.

Regulatory authorities may conduct periodic audits to ensure compliance with this

regulation.

Article 9: Compliance and Audits

Organizations must maintain documentation demonstrating compliance with this

regulation.

Regulatory authorities may conduct periodic audits to ensure compliance with this

regulation.

Article

------------------------------------------------------------

Ask a legal question: What does the regulation say about data privacy?


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
Data Protection Regulation

The data protection regulation is a set of guidelines and procedures for the protection of personal data.

The regulation provides for a framework for the protection of personal data and establishes the principles for

the protection of personal data.

The regulation also provides for a mechanism for the protection of personal data by law enforcement agencies.

The regulation also provides for the protection of personal data by law enforcement agencies.

Article 6: Data Protection Regulation

The data protection regulation is a set of guidelines and procedures for the protection of personal data.

The regulation provides for a framework for the protection of personal data by law enforcement agencies.

The regulation also provides for the protection of personal data by law enforcement

------------------------------------------------------------

Ask a legal question: What penalties are mentioned for non-compliance?


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
Data Protection Regulation (Sample Document)

This sample regulation is created for educational and demonstration purposes. It simulates a legal
document that can be used for testing Retrieval-Augmented Generation (RAG) systems. Theregulation outlines rules related to data collection, storage, processing, cross-border transfers, andpenalties for non■compliance.Data Protection Regulation (Sample Document)This sample regulation is created for educational and demonstration purposes. It simulates a legaldocument that can be used for testing Retrieval-Augmented Generation (RAG) systems.
Article 2: Data Protection Regulation

Organizations must maintain documentation demonstrating compliance with this regulation.

Regulatory authorities may conduct periodic audits to ensure compliance with this regulation.

------------------------------------------------------------

Ask a legal question: exit
Session ended.


In [ ]:
# Scenario: University Library Assistant
# A large university library has thousands of digitized textbooks, research papers, and course notes. Students often struggle to find specific explanations or summaries when preparing for exams. Instead of manually searching through PDFs, the library deploys a RAG chatbot that acts like a study companion.
# How It Works
# - Input Source: Students upload or access a textbook PDF (e.g., Introduction_to_Data_Science.pdf).
# - Chunking: The chatbot splits the textbook into smaller sections so that each concept is searchable.
# - Embeddings + Vector DB: Each section is embedded and stored in Chroma, making the textbook searchable by meaning.
# - Retriever: When a student asks, “Explain the difference between supervised and unsupervised learning,” the retriever pulls the most relevant sections.
# - LLM Response: The Hugging Face model generates a clear, concise answer tailored to the student’s query.
# - Interactive Chat: Students can keep asking follow-up questions, turning the textbook into a conversational tutor.

# ==========================================================
# UNIVERSITY LIBRARY ASSISTANT (RAG CHATBOT)
# ==========================================================

# STEP 1 — Install Libraries
!pip -q install chromadb sentence-transformers pypdf transformers


# STEP 2 — Import Libraries
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline
from google.colab import files


# STEP 3 — Upload Textbook PDF
uploaded = files.upload()
pdf_file = list(uploaded.keys())[0]

print("Loading textbook...")

reader = PdfReader(pdf_file)

text = ""
for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text

print("Textbook Loaded")
print("Total Characters:", len(text))


# STEP 4 — Chunk the Textbook
def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total Chunks Created:", len(chunks))


# STEP 5 — Load Embedding Model
print("Loading embedding model...")

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model ready")


# STEP 6 — Create Vector Database
client = chromadb.Client()

try:
    client.delete_collection("library_collection")
except:
    pass

collection = client.create_collection("library_collection")

print("Vector database ready")


# STEP 7 — Store Chunks in Vector DB
print("Storing textbook sections...")

for i, chunk in enumerate(chunks):

    embedding = embedding_model.encode(chunk).tolist()

    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All sections stored successfully")


# STEP 8 — Load Language Model
print("Loading LLM...")

generator = pipeline(
    "text-generation",
    model="gpt2",
    max_new_tokens=150,
    temperature=0.3
)

print("LLM ready")


# STEP 9 — Retriever Function
def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


# STEP 10 — Answer Function
def answer_question(question):

    docs = retrieve(question)

    context = " ".join(docs)

    prompt = f"""
You are a helpful university study assistant.

Use the following textbook context to answer the student's question.

Context:
{context}

Question:
{question}

Answer in simple terms:
"""

    result = generator(prompt)

    return result[0]["generated_text"].replace(prompt, "")


# STEP 11 — Chat Loop
print("\n===================================")
print("University Library Assistant Ready")
print("Type 'exit' to stop")
print("===================================\n")

while True:

    q = input("Ask a study question: ")

    if q.lower() == "exit":
        print("Session ended.")
        break

    ans = answer_question(q)

    print("\nAnswer:\n", ans)
    print("\n" + "-"*60 + "\n")

Saving Introduction_to_Data_Science (1).pdf to Introduction_to_Data_Science (1).pdf
Loading textbook...
Textbook Loaded
Total Characters: 3427
Total Chunks Created: 8
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model ready
Vector database ready
Storing textbook sections...
All sections stored successfully
Loading LLM...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...


Answer:
 
Data Science is a field of study that combines statistics, computer science, and domain knowledge to extract meaningful insights from data.

1. What is Data Science?

Data Science is an interdisciplinary field that combines statistics, computer science, and domain knowledge to extract meaningful insights from data.

Key Components of Data Science:

cumulative

data

cumulative

cumulative

cumulative

cumulative

cumulative

cumulative

cumulative

cumulative

cumulative

cumulative

cumulative

cumulative

cumulative

cumulative

cumulative

cumulative

cumulative

cumulative

cum

------------------------------------------------------------

Ask a study question: Explain the difference between supervised and unsupervised learning.


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
Unsupervised learning is a learning process that requires a set of inputs and outputs. It is

often called "supervised learning".

Unsupervised learning is a learning process that requires a set of inputs and outputs. It is often called "supervised learning".

The following are examples of unsupervised learning:

A dataset is a collection of data.

A dataset is a collection of data.

A dataset is a collection of data.

A dataset is a collection of data.

A dataset is a collection of data.

A dataset is a collection of data.

A dataset is a collection of data.

A dataset is a collection of data.


------------------------------------------------------------

Ask a study question: How is artificial intelligence used in modern applications?


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
Machine learning is used to perform tasks such as:

- Analyze data,

- Analyze data,

- Analyze data,

- Analyze data,

- Analyze data,

- Analyze data,

- Analyze data,

- Analyze data,

- Analyze data,

- Analyze data,

- Analyze data,

- Analyze data,

- Analyze data,

- Analyze data,

- Analyze data,

- Analyze data,

- Analyze data,

- Analyze data,

- Analyze data,

- Analyze data

------------------------------------------------------------

Ask a study question: exit
Session ended.


In [ ]:
# ==============================================================
# RAG LEGAL COMPLIANCE ASSISTANT
# GRADIO + CHROMA + EMBEDDINGS + LLM
# ==============================================================

# --------------------------------------------------------------
# STEP 1 — Install Dependencies
# --------------------------------------------------------------

!pip install gradio chromadb sentence-transformers pypdf transformers


# --------------------------------------------------------------
# STEP 2 — Import Libraries
# --------------------------------------------------------------

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# --------------------------------------------------------------
# STEP 3 — Load Embedding Model
# --------------------------------------------------------------
# Converts text into vector embeddings for semantic search

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# --------------------------------------------------------------
# STEP 4 — Initialize Vector Database (Chroma)
# --------------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("legal_docs")
except:
    pass

collection = client.create_collection("legal_docs")


# --------------------------------------------------------------
# STEP 5 — Load Language Model
# --------------------------------------------------------------

print("Loading LLM...")

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# --------------------------------------------------------------
# STEP 6 — Document Chunking
# --------------------------------------------------------------

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# --------------------------------------------------------------
# STEP 7 — Process Uploaded PDF
# --------------------------------------------------------------

def process_pdf(file):

    print("Processing PDF...")

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    chunks = chunk_text(text)

    print("Total chunks:", len(chunks))

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Document processed successfully. {len(chunks)} chunks stored."


# --------------------------------------------------------------
# STEP 8 — Retriever
# --------------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    print("\nRetrieved Chunks:\n", docs)

    return docs


# --------------------------------------------------------------
# STEP 9 — Answer Generation
# --------------------------------------------------------------

def answer_question(query):

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are a legal compliance assistant.

Use ONLY the context below to answer the question.

Context:
{context}

Question: {query}

Provide a short clear answer.
"""

    response = llm(
        prompt,
        max_length=200,
        temperature=0.2
    )

    result = response[0]["generated_text"]

    print("\nRaw Model Output:\n", result)

    return result


# --------------------------------------------------------------
# STEP 10 — Chat Function
# --------------------------------------------------------------

def chat(question):

    if not question.strip():
        return "Please enter a question."

    answer = answer_question(question)

    if not answer:
        return "No answer generated."

    return answer


# --------------------------------------------------------------
# STEP 11 — Build Gradio Interface
# --------------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 📜 Legal Compliance RAG Assistant")

    gr.Markdown("""
Upload a legal regulation document and ask questions about:

• compliance rules
• cross-border data transfers
• penalties for violations
• data subject rights
""")

    pdf_input = gr.File(label="Upload Legal PDF")

    upload_button = gr.Button("Process Document")

    status = gr.Textbox(label="Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask a Legal Question"
    )

    answer_box = gr.Textbox(
        label="Answer",
        lines=15
    )

    ask_button = gr.Button("Ask")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# --------------------------------------------------------------
# STEP 12 — Launch Application
# --------------------------------------------------------------

demo.launch()

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM loaded successfully
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a55edf35e49ffa3de2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:

#Scenario: "The Healthcare Policy Navigator"
#Background
#You are part of the Healthcare Compliance & Policy Team at a large hospital network. New government regulations on patient data privacy and telemedicine practices have just been released. The hospital must quickly adapt to ensure compliance and avoid penalties.
#Challenge
#The hospital uploads a PDF of the healthcare regulation into the Policy Navigator (your Gradio + Chroma + LLM app). Your task is to:
#- Process the regulation document so the assistant can store and understand it.
#- Ask compliance-related questions about patient rights, telemedicine rules, and penalties.
#- Generate clear, actionable answers that can guide doctors, administrators, and IT staff


# ==============================================================
# HEALTHCARE POLICY NAVIGATOR (RAG SYSTEM)
# GRADIO + CHROMA + EMBEDDINGS + LLM
# ==============================================================

# --------------------------------------------------------------
# STEP 1 — Install Dependencies
# --------------------------------------------------------------

!pip install gradio chromadb sentence-transformers pypdf transformers


# --------------------------------------------------------------
# STEP 2 — Import Libraries
# --------------------------------------------------------------

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# --------------------------------------------------------------
# STEP 3 — Load Embedding Model
# --------------------------------------------------------------

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# --------------------------------------------------------------
# STEP 4 — Initialize Vector Database
# --------------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("healthcare_docs")
except:
    pass

collection = client.create_collection("healthcare_docs")


# --------------------------------------------------------------
# STEP 5 — Load Language Model (FIXED)
# --------------------------------------------------------------

print("Loading LLM...")

llm = pipeline(
    "text-generation",   # FIXED PIPELINE
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# --------------------------------------------------------------
# STEP 6 — Text Chunking
# --------------------------------------------------------------

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# --------------------------------------------------------------
# STEP 7 — Process Uploaded PDF
# --------------------------------------------------------------

def process_pdf(file):

    print("Processing healthcare regulation document...")

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:
        extracted = page.extract_text()

        if extracted:
            text += extracted

    chunks = chunk_text(text)

    print("Total chunks:", len(chunks))

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Document processed successfully. {len(chunks)} sections stored."


# --------------------------------------------------------------
# STEP 8 — Retrieve Relevant Context
# --------------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    print("\nRetrieved Policy Sections:\n", docs)

    return docs


# --------------------------------------------------------------
# STEP 9 — Generate Answer
# --------------------------------------------------------------

def answer_question(query):

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are a Healthcare Compliance Assistant.

Use ONLY the policy context below to answer.

Context:
{context}

Question: {query}

Give a short and clear answer for hospital staff.
"""

    response = llm(
        prompt,
        max_length=200,
        temperature=0.3
    )

    result = response[0]["generated_text"]

    print("\nModel Output:\n", result)

    return result


# --------------------------------------------------------------
# STEP 10 — Chat Function
# --------------------------------------------------------------

def chat(question):

    if not question.strip():
        return "Please enter a question."

    answer = answer_question(question)

    return answer


# --------------------------------------------------------------
# STEP 11 — Build Gradio Interface
# --------------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 🏥 Healthcare Policy Navigator")

    gr.Markdown("""
Upload a healthcare regulation PDF and ask questions about:

• patient data privacy
• telemedicine rules
• compliance requirements
• penalties for violations
• patient rights
""")

    pdf_input = gr.File(label="Upload Healthcare Regulation PDF")

    upload_button = gr.Button("Process Document")

    status = gr.Textbox(label="Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask a Compliance Question"
    )

    answer_box = gr.Textbox(
        label="Answer",
        lines=10
    )

    ask_button = gr.Button("Ask")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# --------------------------------------------------------------
# STEP 12 — Launch App
# --------------------------------------------------------------

demo.launch(share=True)

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM loaded successfully
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://97dc432ca0f5051b12.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Scenario: "The Environmental Policy Compliance Assistant"

# Background
# You are part of the Sustainability & Environmental Compliance Team at a global manufacturing company.
# New government regulations on carbon emissions, waste disposal, and renewable energy adoption have just been released.
# The company must ensure compliance to avoid fines and reputational damage.

# Challenge
# The company uploads a PDF of the environmental regulation into the Compliance Assistant (Gradio + Chroma + LLM app).
# Your task is to:
# - Process the regulation document so the assistant can store and understand it.
# - Ask compliance-related questions about emission limits, waste management rules, and renewable energy targets.
# - Generate clear, actionable answers that can guide engineers, sustainability officers, and executives.

# Roles
# - Learner (You): Environmental compliance officer using the assistant.
# - Assistant (The RAG App): Provides answers strictly based on uploaded environmental regulations.
# - Stakeholders: Plant managers, sustainability officers, and executives who need concise compliance guidance.

# 🔄 Flow of the Scenario
# - Upload Environmental Regulation PDF
# Example: “National Carbon Emissions Act 2026”.

# - System Processes Document
# - Splits into chunks.
# - Embeds into vector database.
# - Stores for retrieval.

# - Ask Questions
# - “What is the maximum carbon emission allowed per factory per year?”
# - “What penalties apply if hazardous waste is not disposed of properly?”
# - “What renewable energy targets must we meet by 2030?”

# - Assistant Responds
# - Retrieves relevant chunks.
# - Generates compliance-focused answers.
# - Provides short, clear guidance.

# - Outcome
# - Learners practice extracting environmental obligations.
# - Managers receive summarized compliance insights.
# - Executives gain confidence in sustainability strategy alignment.

# 🎯 Training Objective
# This scenario helps learners:
# - Understand how RAG systems can support environmental compliance.
# - Practice formulating precise queries to extract obligations.
# - Experience how AI can simplify complex environmental regulations into actionable steps.

# 👉 Would you like me to also draft a sample regulation PDF text (like the healthcare one I created earlier) for this environmental context, so you can upload it into your assistant and simulate queries?

# ==============================================================
# ENVIRONMENTAL POLICY COMPLIANCE ASSISTANT
# GRADIO + CHROMA + EMBEDDINGS + LLM (RAG SYSTEM)
# ==============================================================

# --------------------------------------------------------------
# STEP 1 — Install Dependencies
# --------------------------------------------------------------

!pip install gradio chromadb sentence-transformers pypdf transformers


# --------------------------------------------------------------
# STEP 2 — Import Libraries
# --------------------------------------------------------------

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# --------------------------------------------------------------
# STEP 3 — Load Embedding Model
# --------------------------------------------------------------

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# --------------------------------------------------------------
# STEP 4 — Initialize Vector Database
# --------------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("environmental_docs")
except:
    pass

collection = client.create_collection("environmental_docs")


# --------------------------------------------------------------
# STEP 5 — Load Language Model
# --------------------------------------------------------------

print("Loading LLM...")

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# --------------------------------------------------------------
# STEP 6 — Text Chunking
# --------------------------------------------------------------

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# --------------------------------------------------------------
# STEP 7 — Process Environmental Policy PDF
# --------------------------------------------------------------

def process_pdf(file):

    print("Processing environmental regulation...")

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted

    chunks = chunk_text(text)

    print("Total chunks:", len(chunks))

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Environmental regulation processed successfully. {len(chunks)} sections stored."


# --------------------------------------------------------------
# STEP 8 — Retrieve Relevant Policy Sections
# --------------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    print("\nRetrieved Policy Sections:\n", docs)

    return docs


# --------------------------------------------------------------
# STEP 9 — Generate Compliance Answer
# --------------------------------------------------------------

def answer_question(query):

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are an Environmental Compliance Assistant.

Use ONLY the environmental regulation context below.

Context:
{context}

Question: {query}

Provide a short, clear, actionable answer for:
- engineers
- sustainability officers
- executives
"""

    response = llm(
        prompt,
        max_length=200,
        temperature=0.3
    )

    result = response[0]["generated_text"]

    print("\nModel Output:\n", result)

    return result


# --------------------------------------------------------------
# STEP 10 — Chat Function
# --------------------------------------------------------------

def chat(question):

    if not question.strip():
        return "Please enter a compliance question."

    return answer_question(question)


# --------------------------------------------------------------
# STEP 11 — Build Gradio Interface
# --------------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 🌱 Environmental Policy Compliance Assistant")

    gr.Markdown("""
Upload an environmental regulation document and ask about:

• Carbon emission limits
• Waste disposal rules
• Renewable energy targets
• Environmental penalties
• Sustainability compliance requirements
""")

    pdf_input = gr.File(label="Upload Environmental Regulation PDF")

    upload_button = gr.Button("Process Regulation Document")

    status = gr.Textbox(label="Processing Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask a Compliance Question"
    )

    answer_box = gr.Textbox(
        label="Assistant Answer",
        lines=12
    )

    ask_button = gr.Button("Ask Assistant")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# --------------------------------------------------------------
# STEP 12 — Launch Application
# --------------------------------------------------------------

demo.launch(share=True)

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM loaded successfully
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5dd3f69d2d54fb7342.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
